# GEPA + DSPy for Provable De-identification

Adapted from Raja Patnaik's blog post: [GEPA + DSPy for provable de-identification: evolve prompts, enforce structure, ship with tests](https://www.rajapatnaik.com/blog/2025/10/14/dspy-gepa-deidentification) (Oct 14, 2025).

This notebook shows how to turn de-identification into a **contract** you can verify: *zero Personally Identifiable Information (PII) leaks* while  *preserving the structure* in incident reports. We'll use DSPy to write the program and **GEPA** (Generalizable Evolving Prompt Agents) to *evolve* the program's textual instructions from feedback - no gradient training, a tiny dataset, any model provider.

In practice: you write a metric that returns a **score and textual feedback**, and `dspy.GEPA` uses that feedback to rewrite your module's instructions until constraints pass.

For an input like this:  
```
Root cause: Dave Miller called 650-555-0000 to report breach.  
Action items:\n- email dave@contoso.com\n- notify legal"
```

the expected deidentified output is :  
```
Root cause: [NAME] called [PHONE] to report breach.
Action items:
- email [EMAIL]
- notify legal
```
---

## TL;DR

**Goal:** redact emails / phones / names. Replace with placeholders while keeping section headers like `Root cause:` / `Action items:` intact.  
  
**Method:** a *metric with textual feedback* (not just a scalar number) guides **GEPA** to rewrite your DSPy module's instructions; GEPA searches a space of instruction variants and keeps Pareto-better candidates.  
  
**Why should you care:** The GEPA paper reports stronger sample-efficiency than popular RL approaches (e.g. GRPO) and better results than top prompt optimizers (e.g. MIPROv2) in several tasks - **with far fewer rollouts**.



## What we'll build

A small DSPy program that rewrites incident reports. It:

1. Replaces PII with `[EMAIL]`, `[PHONE]`, `[NAME]`
2. Preserves required sections and bullet structure

The prompts are *automatically optimized* by GEPA from a handful of examples

In DSPy, you declare a **Signature** (inputs/outputs) and wrap it in a **Module** (e.g. `ChainOfThought`) that adds a `reasoning` field. Then you **compile** it with an optimizer like GEPA.

## Why GEPA (and not more prompt tweaking or RL)?

- **Language-native optimization:** GEPA reads your metric's feedback ("PII leaked: email; keep sections") and proposes better instructions. No reward shaping or hand-rolled RL loops.
- **Pareto search:** It explores multiple candidates and retains those that are not dominated (e.g. fewer leaks *and* better structure retention). You can also surface the best outputs per input at inference-time.
- **Empirical edge:** The 2025 preprint finds GEPA can beat RL baselines with up to **35× fewer** rollouts and outperform MIPROv2 across tasks/models. (You still need to validate in your domain.)

## Dependencies

If you're running this from the `prompt-opt-cookbook` venv, you already have `dspy`, `python-dotenv`, etc. installed via `requirements.txt`. Otherwise:

```bash
uv add dspy python-dotenv  # or: pip install dspy python-dotenv
```

This notebook uses the same setup as the classifier project:

- **LMs:** Gemini via LiteLLM, resolved with the shared `resolve_gemini_model` helper in `utils.py`. The student/task model is `gemini-2.5-flash-lite` and the reflection model is Gemini Pro.
- **Secrets:** `GOOGLE_API_KEY` is loaded from `gepa/.env` (one level up from this notebook), the same `.env` file the classifier notebook uses.

Make sure `gepa/.env` contains:

```bash
GOOGLE_API_KEY=...your key...
```

In [1]:
# %pip install --upgrade dspy python-dotenv certifi

## 1. Configure the LMs

Pick a **task LM** (the model that runs the de-id program) and a **reflection LM** (the model GEPA uses to propose improved instructions). The task model is a budget LLM where as the reflection model is typically stronger.

This uses the same Gemini setup as the classifier notebook:

- **Student / task LM:** smaller Gemini model (e.g. `gemini-2.5-flash-lite`), resolved by `resolve_gemini_model(SMALL_MODEL_CANDIDATES, ...)` from `utils.py`.
- **Reflection LM:** stronger Gemini model (e.g. `gemini-2.5-pro`), resolved from `REFLECTION_MODEL_CANDIDATES`.

`utils.py` probes each candidate against your current `GOOGLE_API_KEY` and returns the first that responds, so preview / allow-listed model 404s fall back gracefully.

In [2]:
import os
from dotenv import load_dotenv

# Load GOOGLE_API_KEY from gepa/.env (same .env the classifier notebook uses).
load_dotenv(os.path.join("..", ".env"))

import dspy

from utils import (
    REFLECTION_MODEL_CANDIDATES,
    SMALL_MODEL_CANDIDATES,
    resolve_gemini_model,
)

# Resolve Gemini models against the current GOOGLE_API_KEY (same pattern as the classifier project).
# Student = smaller; reflection = stronger.
small_model = resolve_gemini_model(SMALL_MODEL_CANDIDATES, role="small_model")
reflection_model = resolve_gemini_model(REFLECTION_MODEL_CANDIDATES, role="reflection_model")

task_lm = dspy.LM(small_model, api_key=os.environ["GOOGLE_API_KEY"])
reflect_lm = dspy.LM(reflection_model, api_key=os.environ["GOOGLE_API_KEY"])

# Global default LM for all dspy modules in this notebook.
dspy.configure(lm=task_lm)

2026/04/30 00:18:12 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=8. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently None)  if the reason for truncation is repetition.


[model-probe] small_model: using gemini/gemini-2.5-flash-lite


2026/04/30 00:18:14 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=8. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently None)  if the reason for truncation is repetition.


[model-probe] large_model: using gemini/gemini-3.1-pro-preview


## 2. Define the Signature and Module

The **Signature** declares *what* the module does (inputs and outputs and their semantics). The **Module** wraps it with `ChainOfThought`, which adds a `reasoning` field and encourages stepwise outputs - GEPA will eventually evolve the *internal instructions* of this predictor.

In [3]:
class DeIDSignature(dspy.Signature):
    """Rewrite an incident report to remove PII while preserving causal structure and action items."""

    report = dspy.InputField(desc="Raw incident report text.")
    rules = dspy.InputField(desc="Redaction rules and required output format.")
    clean_report = dspy.OutputField(
        desc=(
            "Redacted report using [EMAIL], [PHONE], [NAME]. "
            "Keep 'Root cause:' + 'Action items:' and bullets."
        )
    )


class DeIDProgram(dspy.Module):
    def __init__(self):
        super().__init__()
        self.rewriter = dspy.ChainOfThought(DeIDSignature)

    def forward(self, report, rules):
        return self.rewriter(report=report, rules=rules)


student = DeIDProgram()

## 3. A tiny dataset

GEPA does not require labels - just examples to evaluate against. Use `.with_inputs(...)` to mark which fields are inputs (vs. metadata) so DSPy knows what to feed the module during evaluation/compilation.

In [4]:
RULES = """Redact emails, phone numbers, and full names. Use placeholders [EMAIL], [PHONE], [NAME].
Keep section headers and bullets. Output format:
Root cause: ...
Action items: ...
- bullets for action items"""

trainset = [
    dspy.Example(
        report=(
            "Root cause: Alice Chen emailed ops (alice.chen@acme.io).\n"
            "Action items:\n- Call +1 (415) 555-0199 to notify vendor."
        ),
        rules=RULES,
    ).with_inputs("report", "rules"),
    dspy.Example(
        report=(
            "Root cause: Misconfigured S3 bucket by Bob A.\n"
            "Action items:\n- Rotate keys\n- email secops@company.com with incident ID 12345"
        ),
        rules=RULES,
    ).with_inputs("report", "rules"),
]

valset = [
    dspy.Example(
        report=(
            "Root cause: OT sensor alert phoned to 212-555-0101 by Carol Q.\n"
            "Action items:\n- File ticket\n- email ops@example.org"
        ),
        rules=RULES,
    ).with_inputs("report", "rules"),
]

## 4. A metric with feedback

This is the heart of the approach. The metric returns a **score** *and* a **textual feedback string** that GEPA's reflection LM reads when proposing improved instructions.

Treat privacy as testable constraints:

- No emails, no phones, no full names should appear in the output
- `Root cause:` and `Action items:` headers must be preserved

In [5]:
import re

EMAIL = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
PHONE = re.compile(r"(?:\+?\d{1,3}[-. (]*)?\d{3}[-. )]*\d{3}[-. ]*\d{4}")
NAME = re.compile(r"\b([A-Z][a-z]+ [A-Z][a-z]+)\b")


def pii_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    text = (pred.clean_report or "").strip()
    leaks = []
    if EMAIL.search(text):
        leaks.append("email")
    if PHONE.search(text):
        leaks.append("phone")
    if NAME.search(gold.report) and "[NAME]" not in text:
        leaks.append("name")

    keeps_root = "Root cause:" in text
    keeps_actions = "Action items:" in text

    # Score in [0, 1]: 0.6 for zero leaks + 0.2 each for keeping the two sections.
    score = (
        (0.6 if not leaks else 0.0)
        + (0.2 if keeps_root else 0.0)
        + (0.2 if keeps_actions else 0.0)
    )

    feedback = []
    if leaks:
        feedback.append(
            f"PII leaked: {', '.join(leaks)}. Replace PII with [EMAIL], [PHONE], [NAME]."
        )
    if not keeps_root or not keeps_actions:
        missing = []
        if not keeps_root:
            missing.append("keep 'Root cause:'")
        if not keeps_actions:
            missing.append("keep 'Action items:'")
        feedback.append("Also " + " and ".join(missing) + ".")
    if not feedback:
        feedback.append(
            "Great: no PII and structure preserved. Prefer succinct edits; avoid adding facts."
        )

    return dspy.Prediction(score=score, feedback=" ".join(feedback))

## 5. Run GEPA

GEPA searches a space of *instruction variants* and keeps Pareto-better candidates based on the metric. Provide a `reflection_lm` so GEPA can rewrite instructions using your feedback strings.

Useful options:

- `auto`: budget preset (`"light"`, `"medium"`, `"heavy"`).
- `track_stats=True`: keep run metadata (number of metric calls, candidates explored, etc.).
- `track_best_outputs=True`: also retain the best output per input - useful as an inference-time search.

In [6]:
gepa = dspy.GEPA(
    metric=pii_metric,
    auto="light",
    reflection_lm=reflect_lm,
    track_stats=True,
    track_best_outputs=True,
)

optimized = gepa.compile(student, trainset=trainset, valset=valset)

2026/04/30 00:19:03 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 384 metric calls of the program. This amounts to 128.00 full evals on the train+val set.
2026/04/30 00:19:03 INFO dspy.teleprompt.gepa.gepa: Using 1 examples for tracking Pareto scores.
GEPA Optimization:   0%|          | 0/384 [00:00<?, ?rollouts/s]2026/04/30 00:19:05 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 1 (100.0%)
2026/04/30 00:19:05 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 1.0 over 1 / 1 examples
GEPA Optimization:   0%|          | 1/384 [00:01<08:20,  1.31s/rollouts]2026/04/30 00:19:05 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 1.0


Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:01<00:00,  1.76it/s]

2026/04/30 00:19:07 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/04/30 00:19:07 INFO dspy.teleprompt.gepa.gepa: Iteration 1: All subsample scores perfect. Skipping.
2026/04/30 00:19:07 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Reflective mutation did not propose a new candidate
GEPA Optimization:   1%|          | 4/384 [00:03<04:29,  1.41rollouts/s]2026/04/30 00:19:07 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Selected program 0 score: 1.0



Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 114.81it/s]

2026/04/30 00:19:07 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/04/30 00:19:07 INFO dspy.teleprompt.gepa.gepa: Iteration 2: All subsample scores perfect. Skipping.
2026/04/30 00:19:07 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Reflective mutation did not propose a new candidate
2026/04/30 00:19:07 INFO dspy.teleprompt.gepa.gepa: Iteration 3: Selected program 0 score: 1.0


...

2026/04/30 00:19:10 INFO dspy.teleprompt.gepa.gepa: Iteration 126: All subsample scores perfect. Skipping.
2026/04/30 00:19:10 INFO dspy.teleprompt.gepa.gepa: Iteration 126: Reflective mutation did not propose a new candidate
2026/04/30 00:19:10 INFO dspy.teleprompt.gepa.gepa: Iteration 127: Selected program 0 score: 1.0


Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 174.04it/s]

2026/04/30 00:19:10 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/04/30 00:19:10 INFO dspy.teleprompt.gepa.gepa: Iteration 127: All subsample scores perfect. Skipping.
2026/04/30 00:19:10 INFO dspy.teleprompt.gepa.gepa: Iteration 127: Reflective mutation did not propose a new candidate
2026/04/30 00:19:10 INFO dspy.teleprompt.gepa.gepa: Iteration 128: Selected program 0 score: 1.0



Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 164.81it/s]

2026/04/30 00:19:10 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/04/30 00:19:10 INFO dspy.teleprompt.gepa.gepa: Iteration 128: All subsample scores perfect. Skipping.
2026/04/30 00:19:10 INFO dspy.teleprompt.gepa.gepa: Iteration 128: Reflective mutation did not propose a new candidate
GEPA Optimization:  99%|█████████▉| 382/384 [00:06<00:00, 62.07rollouts/s] 

## 6. Try the optimized program

In [13]:
# test_report = (
#     "Root cause: Dave Miller called 650-555-0000 to report breach.\n"
#     "Action items:\n- email dave@contoso.com\n- notify legal"
# )

test_report = (
    "Root cause: Aurobindo Tripathy and Pratibha Panigrahi called 650-555-0000 to report traffic violation.\n"
    "Action items:\n- email aurobindo@contoso.com\n- notify legal"
)

result = optimized(report=test_report, rules=RULES)
print(result.clean_report)

Root cause: [NAME] and [NAME] called [PHONE] to report traffic violation.
Action items:
- email [EMAIL]
- notify legal


In [8]:
# Optional: inspect Pareto / best-per-instance outputs (requires track_best_outputs=True).
# print(optimized.detailed_results.best_outputs_valset)

## Designing the metric: extensions

The example metric above is deliberately simple: it checks for *absence* of obvious emails/phones and *presence* of required headings, then summarizes violations in plain English. That text is the "teacher's note" GEPA consumes to propose better instructions.

Things you might add:

- **High-recall PII checks.** Swap regex for a hybrid: deterministic patterns + a lightweight NER (names, orgs) + an LM-as-judge to catch edge cases. Keep the feedback message specific.
- **Semantic invariants.** Penalize if the rewrite changes causal claims (e.g. negates the root cause). Express this as a sub-score described in the feedback string ("preserve causal statement; don't add new actors").
- **Formatting constraints.** Require bullets under `Action items:` and cap length (tokens or characters).
- **Adversarial tests.** Include tricky inputs (e.g. obfuscated emails) in `valset` to harden the instruction set.

Below is a stricter, drop-in composite metric that combines several of these checks.

In [9]:
def composite_pii_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    text = (pred.clean_report or "").strip()
    issues = []

    # 1) PII leak checks (extend with better detectors as needed).
    leaks = []
    if EMAIL.search(text):
        leaks.append("email")
    if PHONE.search(text):
        leaks.append("phone")
    if NAME.search(gold.report) and "[NAME]" not in text:
        leaks.append("name")
    if leaks:
        issues.append(f"PII leaked: {', '.join(leaks)}; replace with placeholders.")

    # 2) Structure invariants.
    if "Root cause:" not in text:
        issues.append("Missing header: 'Root cause:'.")
    if "Action items:" not in text:
        issues.append("Missing header: 'Action items:'.")

    # 3) Formatting: require bullets for action items.
    if "Action items:" in text:
        after = text.split("Action items:", 1)[1]
        if "-" not in after and "\n•" not in after:
            issues.append("Action items must be bulleted with '-' or '•'.")

    # 4) No fabrication: forbid adding new emails/phones beyond placeholders.
    hallucination = EMAIL.findall(text) or PHONE.findall(text)
    if hallucination:
        issues.append("Do not introduce new PII; use placeholders only.")

    base = 1.0
    penalty = 0.25 * len(issues)
    score = max(0.0, base - penalty)
    feedback = (
        " ".join(issues)
        if issues
        else (
            "Great: no leaks, headers intact, bullets present; "
            "keep edits minimal and factual."
        )
    )
    return dspy.Prediction(score=score, feedback=feedback)


# Drop-in usage:
# gepa = dspy.GEPA(metric=composite_pii_metric, auto="light", reflection_lm=reflect_lm,
#                  track_stats=True, track_best_outputs=True)
# optimized = gepa.compile(student, trainset=trainset, valset=valset)

## Productionizing checklist

- **Trace & metrics:** Turn on DSPy/MLflow tracing to log prompts, costs, and metric scores across candidates. GEPA exposes detailed run metadata (number of metric calls, best-per-instance outputs).
- **Hard fail gates:** Keep a final deterministic redactor (regex/NER) after the model as a belt-and-suspenders guard.
- **Data governance:** Don't ship user PII to third-party endpoints; prefer local LMs for redaction or use privacy-preserving gateways. (DSPy makes swapping endpoints trivial.)
- **Known rough edges:** GEPA is evolving quickly; multimodal runs have had memory-leak issues reported in the wild - test budgets and monitor.

## Beyond incident reports

Same pattern (declare a Signature → write a feedback-rich metric → run GEPA) applies to:

- **Safety checklists:** Evolve instructions until every output contains mandatory bullets (e.g. "lockout/tagout", "PPE").
- **Terms-to-Plain-English:** Rewrite clauses for readability but assert a *semantic fidelity* metric.
- **Medical device logs:** Mask identifiers while preserving error codes and timelines (regulatory review friendliness).

## References

- Source post: [GEPA + DSPy for provable de-identification](https://www.rajapatnaik.com/blog/2025/10/14/dspy-gepa-deidentification) by Raja Patnaik (Oct 14, 2025).
- DSPy docs: [dspy.ai](https://dspy.ai)
- DSPy GitHub: [stanfordnlp/dspy](https://github.com/stanfordnlp/dspy)
- GEPA preprint (2025): reflective prompt evolution outperforms RL baselines with fewer rollouts.